<a href="https://colab.research.google.com/github/Carrinson/collab-files/blob/main/Regression_with_Keras_py_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [115]:
# All Libraries required for this lab are listed below. The libraries pre-installed on Skills Network Labs are commented.
# If you run this notebook on a different environment, e.g. your desktop, you may need to uncomment and install certain libraries.
%pip install numpy==2.0.2
%pip install pandas==2.2.2
%pip install tensorflow keras


In [116]:
import pandas as pd
import numpy as np
import keras
from sklearn.preprocessing import LabelEncoder


import warnings
warnings.simplefilter('ignore', FutureWarning)

In [117]:
filepath='/content/nigeria_houses_data.csv'
houses_data = pd.read_csv(filepath)

houses_data.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
0,6.0,5.0,5.0,4.0,Detached Duplex,Mabushi,Abuja,450000000.0
1,4.0,5.0,5.0,4.0,Terraced Duplexes,Katampe,Abuja,800000000.0
2,4.0,5.0,5.0,4.0,Detached Duplex,Lekki,Lagos,120000000.0
3,4.0,4.0,5.0,6.0,Detached Duplex,Ajah,Lagos,40000000.0
4,4.0,4.0,5.0,2.0,Semi Detached Duplex,Lekki,Lagos,75000000.0


In [118]:
le = LabelEncoder()
houses_data['title']  = le.fit_transform(houses_data['title'] ) + 1
houses_data['town']  = le.fit_transform(houses_data['town'] ) + 1
houses_data['state']  = le.fit_transform(houses_data['state'] ) + 1

In [119]:
houses_data.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
0,6.0,5.0,5.0,4.0,3,128,2,450000000.0
1,4.0,5.0,5.0,4.0,7,111,2,800000000.0
2,4.0,5.0,5.0,4.0,3,123,18,120000000.0
3,4.0,4.0,5.0,6.0,3,11,18,40000000.0
4,4.0,4.0,5.0,2.0,5,123,18,75000000.0


In [120]:
houses_data.shape

(24326, 8)

In [121]:
houses_data.describe()

,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
count,24326.000000,24326.000000,24326.000000,24326.000000,24326.000000,24326.000000,24326.000000,2.432600e+04
mean,4.338814,4.600798,5.176355,4.041725,3.557552,101.793143,15.711625,3.013802e+08
std,1.138497,1.163161,1.226253,1.399936,1.644986,44.186951,6.024042,1.220403e+10
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,9.000000e+04
25%,4.000000,4.000000,5.000000,4.000000,3.000000,78.000000,18.000000,5.200000e+07
50%,4.000000,5.000000,5.000000,4.000000,3.000000,123.000000,18.000000,8.500000e+07
75%,5.000000,5.000000,6.000000,4.000000,4.000000,123.000000,18.000000,1.600000e+08
max,9.000000,9.000000,9.000000,9.000000,7.000000,189.000000,25.000000,1.800000e+12


In [122]:
houses_data.isnull().sum()

,0
bedrooms,0
bathrooms,0
toilets,0
parking_space,0
title,0
town,0
state,0
price,0


In [123]:
houses_data_columns = houses_data.columns
print(houses_data_columns)

Index(['bedrooms', 'bathrooms', 'toilets', 'parking_space', 'title', 'town',
       'state', 'price'],
      dtype='object')


In [124]:
houses_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24326 entries, 0 to 24325
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   bedrooms       24326 non-null  float64
 1   bathrooms      24326 non-null  float64
 2   toilets        24326 non-null  float64
 3   parking_space  24326 non-null  float64
 4   title          24326 non-null  int64  
 5   town           24326 non-null  int64  
 6   state          24326 non-null  int64  
 7   price          24326 non-null  float64
dtypes: float64(5), int64(3)
memory usage: 1.5 MB


In [125]:
predictors = houses_data[houses_data_columns[houses_data_columns != 'price']] # all columns except Price
target = houses_data['price'] # Price column

<a id="item2"></a>


In [126]:
predictors.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state
0,6.0,5.0,5.0,4.0,3,128,2
1,4.0,5.0,5.0,4.0,7,111,2
2,4.0,5.0,5.0,4.0,3,123,18
3,4.0,4.0,5.0,6.0,3,11,18
4,4.0,4.0,5.0,2.0,5,123,18


In [127]:
target.head()

,price
0,450000000.0
1,800000000.0
2,120000000.0
3,40000000.0
4,75000000.0


In [128]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state
0,1.459103,0.343205,-0.143816,-0.029805,-0.338940,0.593090,-2.276150
1,-0.297598,0.343205,-0.143816,-0.029805,2.092692,0.208361,-2.276150
2,-0.297598,0.343205,-0.143816,-0.029805,-0.338940,0.479935,0.379874
3,-0.297598,-0.516521,-0.143816,1.398832,-0.338940,-2.054750,0.379874
4,-0.297598,-0.516521,-0.143816,-1.458441,0.876876,0.479935,0.379874


In [129]:
n_cols = predictors_norm.shape[1] # number of predictors

In [130]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

<a id='item33'></a>


In [131]:
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))

    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [132]:
# build the model
model = regression_model()

In [133]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
533/533 - 6s - 11ms/step - loss: 212653430111226298368.0000 - val_loss: 557783554440822784.0000
Epoch 2/100
533/533 - 3s - 5ms/step - loss: 212653377334668165120.0000 - val_loss: 557783554440822784.0000
Epoch 3/100
533/533 - 4s - 8ms/step - loss: 212653482887784431616.0000 - val_loss: 557783520081084416.0000
Epoch 4/100
533/533 - 1s - 3ms/step - loss: 212653394926854209536.0000 - val_loss: 557783520081084416.0000
Epoch 5/100
533/533 - 2s - 4ms/step - loss: 212653412519040253952.0000 - val_loss: 557783348282392576.0000
Epoch 6/100
533/533 - 1s - 3ms/step - loss: 212653412519040253952.0000 - val_loss: 557783313922654208.0000
Epoch 7/100
533/533 - 1s - 2ms/step - loss: 212653359742482120704.0000 - val_loss: 557783142123962368.0000
Epoch 8/100
533/533 - 1s - 2ms/step - loss: 212653377334668165120.0000 - val_loss: 557783004685008896.0000
Epoch 9/100
533/533 - 1s - 2ms/step - loss: 212653465295598387200.0000 - val_loss: 557782867246055424.0000
Epoch 10/100
533/533 - 1s - 2ms/step